# Criterium Research API Tutorial
This notebook walks through the Gemini-powered research API.

**Prerequisites:**
1. Ensure your server is running (`make run` in a terminal).
2. Ensure your `.env` file contains a valid Gemini API key. A Firecrawl key is only needed when using the alternative Firecrawl researcher.

In [ ]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"

def print_json(data):
    # default=str handles non-JSON-native types like datetime
    print(json.dumps(data, indent=2, default=str))

## 1. Create a Research Collection
Criterium uses a file-backed SQLite database by default, so collections, jobs, and completed research survive server restarts. This tutorial creates a Book Series collection that can be reused in later sessions.

Every property is automatically required, including properties of nested objects. If research cannot establish a value reliably, the result contains an explicit `null` rather than omitting the property or guessing.

Because the client and server share `schemas`, we can build the schema using Pydantic rather than writing raw JSON manually.

In [ ]:
from criterium import schemas

schema = schemas.ResearchSchemaObject(
  properties={
    "title": schemas.ResearchSchemaString(
      description="The exact title of the primary book.",
    ),
    "year": schemas.ResearchSchemaInteger(
      description="The 4-digit year the book was published.",
    ),
    "series_books": schemas.ResearchSchemaArray(
      description="Other books in the same series.",
      items=schemas.ResearchSchemaObject(
        properties={
          "title": schemas.ResearchSchemaString(),
          "chronological_order": schemas.ResearchSchemaInteger()
        }
      )
    )
  },
)

payload = schemas.CollectionCreate(
    name="Book Series Researcher",
    extraction_prompt="You are a literary researcher. Extract the details of the primary book discussed on this page, as well as a list of any other books mentioned that belong to the same series.",
    research_schema=schema
)

try:
    response = requests.post(f"{BASE_URL}/collections", json=payload.model_dump(exclude_none=True))
    response.raise_for_status()
    
    collection_data = schemas.CollectionResponse.model_validate(response.json())
    collection_id = collection_data.id
    print(f"Created collection with ID: {collection_id}")
    print_json(collection_data.model_dump())
    
except requests.exceptions.ConnectionError:
    print("Error: Could not connect to the server at", BASE_URL)
    print("Please make sure your server is running in a terminal with: make run")
except Exception as e:
    print("Request failed:", e)

## 2. Verify / List All Collections
Now let's query `GET /collections` to verify our newly created collection is listed in the database.

In [ ]:
response = requests.get(f"{BASE_URL}/collections")

# Parse list of Pydantic models
collections = [schemas.CollectionResponse.model_validate(c) for c in response.json()]

print(f"Found {len(collections)} collections.")
for col in collections:
    print(f"- {col.id}: {col.name}")

## 3. Queue Product Research
Research runs as durable background jobs. One request can queue up to 100 unique products and returns immediately with `202 Accepted`. Each returned job can then be monitored independently until it succeeds or fails.

Three workers process research concurrently by default. The `RESEARCH_WORKER_CONCURRENCY` environment setting can tune this from 1 to 16.

Behind the scenes, the API and workers will:
1. Create one queued job per product in a single transaction
2. Claim and research products concurrently
3. Extract schema-constrained data and capture cited references
4. Save each successful product and mark its job complete

In [ ]:
import time

# This requires a valid GEMINI_API_KEY in the root .env file
research_request = schemas.ResearchJobsCreate(
    product_infos=[
        "The Fellowship of the Ring book",
        "The Two Towers book",
        "The Return of the King book",
    ]
)

print(f"Queueing {len(research_request.product_infos)} research jobs...")
response = requests.post(
    f"{BASE_URL}/collections/{collection_id}/research-jobs",
    json=research_request.model_dump(),
)
response.raise_for_status()

jobs = [schemas.ResearchJobResponse.model_validate(item) for item in response.json()]
job_ids = {job.id for job in jobs}
print("Queued jobs:", sorted(job_ids))

while any(job.status in {"queued", "running"} for job in jobs):
    time.sleep(1)
    response = requests.get(f"{BASE_URL}/collections/{collection_id}/research-jobs")
    response.raise_for_status()
    jobs = [
        schemas.ResearchJobResponse.model_validate(item)
        for item in response.json()
        if item["id"] in job_ids
    ]
    print(", ".join(f"{job.product_info}: {job.stage or job.status}" for job in jobs))

failed_jobs = [job for job in jobs if job.status == "failed"]
if failed_jobs:
    for job in failed_jobs:
        print(f"Research failed for {job.product_info}: {job.error_message}")

completed_product_ids = [job.product_id for job in jobs if job.status == "succeeded"]
print("Completed product IDs:", completed_product_ids)

## 4. View the Extracted Data
Once the job succeeds, fetch the collection's products to inspect the grounded research, structured output, and references.

In [ ]:
products_response = requests.get(f"{BASE_URL}/collections/{collection_id}/products")
products_response.raise_for_status()

products = [schemas.ProductResponse.model_validate(p) for p in products_response.json()]

for product in products:
    print("=" * 50)
    print("PRODUCT:", product.name)
    print("PRIMARY SOURCE:", product.source_url)
    print("-" * 50)
    print("EXTRACTED DATA:")
    print_json(product.extracted_data)
    print("REFERENCES:")
    for reference in product.references:
        print(f"- {reference.title or reference.url} ({reference.provider})")
        print(f"  {reference.url}")
    print("=" * 50)